In [ ]:
pip install -U langchain langchain_openai newspaper3k gradio

In [ ]:
import os
import gradio as gr
import newspaper
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

# Set up OpenRouter API key
os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# Initialize LangChain Chat Model
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.5)

def summarize_news(url):
    """Fetches and summarizes a news article given a URL."""
    try:
        # Extract article content
        article = newspaper.Article(url)
        article.download()
        article.parse()

        if not article.text:
            return "❌ Failed to extract article content. Check the URL."

        # System message for AI behavior
        system_prompt = "You are a news summarization assistant. Summarize the article in simple and concise language.The output should be have atleast 200 words and split into atleast 3 paragraphs."

        # Generate summary using OpenRouter API
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"Summarize this article:\n\n{article.text}")
        ])

        return response.content  # Return summarized content

    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# Gradio UI
iface = gr.Interface(
    fn=summarize_news,
    inputs=gr.Textbox(label="Enter Article URL"),
    outputs=gr.Textbox(label="Article Summary"),
    title="📰 AI News Summarizer",
    description="Enter a news article link, and this AI will summarize it for you using OpenRouter GPT-4.",
)

# Launch Gradio UI
iface.launch(share=True)
